# 🧱 Modul 09: Hyperparameter Tuning, Pipeline & Serialisasi Model
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/09_Pipeline_and_Model_Export.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & alur kerja rekayasa nyata sebelum menyentuh baris kode algoritma.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── 🎯 [5. EVALUATION & TUNING] ◄── [4. Modeling]
```
Banyak pemula merasa pekerjaannya selesai saat model menghasilkan akurasi tinggi di dalam sel notebook. Padahal, **model yang terkunci di dalam memori notebook tidak bisa digunakan oleh siapa pun di dunia nyata**!  
Di **Modul 09**, kita bertransformasi menjadi **Machine Learning Engineer Profesional**: Mengoptimalkan performa model dengan *Hyperparameter Tuning*, menyatukan rantai prapemrosesan ke dalam *Scikit-Learn Pipeline*, dan mengekspor model menjadi file biner (`.joblib`) yang siap dipasang di aplikasi web!


## 📖 Kamus Istilah Awam (Jargon Buster Modul 09)

Sebelum melangkah ke dunia rekayasa machine learning, mari samakan bahasa teknis menggunakan analogi sederhana:

| Istilah Teknis | Bahasa Manusia / Analogi Awam | Mengapa Sangat Krusial? |
| :--- | :--- | :--- |
| **K-Fold Cross-Validation** | Sistem ujian bergilir ($K$ putaran) di mana setiap bagian data merasakan giliran menjadi bahan ajar dan bahan ujian. | Mencegah ilusi keberuntungan di mana model hanya kebetulan pintar di satu potongan data uji acak. |
| **Fold (Lipatan)** | Satu fraksi potongan data (misal: 5-Fold berarti data diiris menjadi 5 bagian yang masing-masing berukuran 20%). | Unit pembagian data untuk simulasi ujian bergilir. |
| **Stratified K-Fold** | Ujian bergilir yang menjamin proporsi kelas target (seimbang vs langka) tetap identik di setiap lipatan. | Wajib digunakan pada kasus data timpang (*imbalanced data*) seperti deteksi penyakit langka atau penipuan. |
| **Parameter** | Bobot internal yang dipelajari mesin sendiri dari data (contoh: koefisien kemiringan $\beta$ pada regresi). | Hasil akhir proses belajar model yang disesuaikan secara otomatis oleh algoritma. |
| **Hyperparameter** | Tombol setelan di luar kabin yang wajib kita atur sebelum mesin mulai belajar (misal: `n_estimators`, `max_depth`). | Menentukan strategi belajar model agar tidak mengalami *overfitting* atau *underfitting*. |
| **GridSearchCV** | Mencoba seluruh kombinasi tombol setelan secara lengkap dan sistematis seperti membuka gembok kombinasi angka. | Menjamin menemukan konfigurasi terbaik di dalam kisi pencarian yang kita tetapkan. |
| **RandomizedSearchCV** | Mengambil sampel kombinasi setelan secara acak cerdas dalam batas percobaan tertentu. | Menghemat waktu komputasi hingga 10x lipat dengan hasil performa yang hampir setara. |
| **Data Leakage (Kebocoran Data)** | Kesalahan fatal pemula saat informasi dari data uji ikut terbaca selama tahap prapemrosesan data latih. | Menyebabkan model tampak berakurasi 99% di laboratorium, namun gagal total saat dipakai di dunia nyata. |
| **Scikit-Learn Pipeline** | Lini ban berjalan otomatis yang menyatukan Imputer $\rightarrow$ Scaler $\rightarrow$ Model menjadi satu kesatuan utuh. | Menjamin nol kebocoran data (*zero leakage*) dan membuat kode produksi rapi dalam satu baris. |
| **ColumnTransformer** | Pengatur jalur pipa yang membedakan perlakuan kolom angka (*imputer + scaler*) vs kolom teks (*imputer + one-hot*). | Menangani dataset dunia nyata yang memuat campuran berbagai tipe data sekaligus. |
| **Serialisasi (`.joblib`)** | Membekukan seluruh isi pipeline ke dalam satu file biner di hard disk agar siap dipindahkan ke server web. | Memungkinkan model dipakai di aplikasi web tanpa perlu melatih ulang dari awal. |
| **Inference (Inferensi)** | Menggunakan file model yang sudah dibekukan untuk memprediksi data pelanggan baru secara seketika (*real-time*). | Titik temu di mana kecerdasan buatan melayani kebutuhan bisnis nyata. |


## 🏭 1. Filosofi Menuju Level Profesional: Mengapa Notebook Saja Tidak Cukup?

### 🍳 Analogi Dapur Restoran vs Pabrik Makanan Siap Saji
* **Notebook (Dapur Manual)**: Di sel 1 Anda mengisi nilai hilang, di sel 2 Anda melakukan one-hot encoding, di sel 3 Anda scaling, lalu di sel 4 Anda melatih model. Jika ada pelanggan baru yang mengirim data, Anda harus menulis ulang seluruh langkah manual tersebut. Ini sangat rentan salah langkah (*human error*) dan kebocoran informasi (*Data Leakage*).
* **Pipeline (Lini Pabrik Otomatis)**: Kita merakit sebuah ban berjalan (*assembly line*). Data mentah kotor masuk di ujung kiri, otomatis dibersihkan, di-skala, dan di ujung kanan langsung keluar prediksi instan!

---

### ⚠️ Dosa Terbesar Machine Learning: Jebakan Kebocoran Data (*Data Leakage Trap*)
Bayangkan skenario pemula berikut:
1. Anda punya 1.000 data. Anda langsung menjalankan `StandardScaler().fit_transform(X)` pada SELURUH 1.000 data.
2. Baru setelah itu Anda membagi data menjadi 800 data latih dan 200 data uji (`train_test_split`).
3. **Mengapa ini adalah DOSA BESAR?** Karena nilai rata-rata (*mean*) dan standar deviasi dari 200 data uji **sudah diintip dan bocor** ke dalam rumus scaling data latih!
4. Model Anda tampak hebat saat ujian karena sudah mencium bocoran data uji, tetapi ketika di-deploy ke server untuk melayani pelanggan nyata, akurasinya anjlok drastis!
5. **Solusi Penyelamat**: Dengan `Pipeline`, Scikit-Learn menjamin operasi `.fit()` **HANYA** dijalankan pada data latih di setiap putaran, dan operasi `.transform()` diterapkan secara terisolasi pada data uji. **Nol Kebocoran Data (*Zero Data Leakage*)!**


## 🔄 2. Validasi Silang (*K-Fold Cross-Validation*): Menghindari Bias Ujian Acak

### 🎓 Analogi Ujian Bergilir
Jika kita hanya membagi data 1 kali (`train_test_split`), model kita bisa saja kebetulan mendapatkan soal ujian yang sangat gampang (sehingga akurasi menipu tinggi) atau kebetulan soalnya sangat aneh (sehingga akurasi anjlok).

**K-Fold Cross-Validation (K=5)** membagi data latih menjadi 5 bagian (*folds*):
* **Putaran 1**: Fold 1, 2, 3, 4 dipakai belajar ➔ Fold 5 dipakai ujian.
* **Putaran 2**: Fold 1, 2, 3, 5 dipakai belajar ➔ Fold 4 dipakai ujian.
* ...dan seterusnya bergilir sampai 5 putaran!
* Nilai akhir adalah **rata-rata skor dari ke-5 putaran**, memberikan estimasi performa model yang jauh lebih objektif dan tahan banting.

> 💡 **Catatan Emas Praktisi**: Untuk data klasifikasi dengan target yang tidak seimbang (misal: 95% transaksi normal vs 5% fraud), gunakan **`StratifiedKFold`** agar proporsi kelas target di setiap lipatan tetap terjaga seimbang!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Memuat dataset standar klasifikasi
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Inisialisasi K-Fold (5 Lipatan Acak Seimbang)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_baseline = RandomForestClassifier(n_estimators=50, random_state=42)

# Hitung Skor Validasi Silang
skor_cv = cross_val_score(rf_baseline, X, y, cv=kfold, scoring='accuracy')

print("=== HASIL 5-FOLD STRATIFIED CROSS VALIDATION ===")
for i, skor in enumerate(skor_cv, 1):
    print(f"Putaran {i} : Akurasi {skor*100:.2f}%")
    
print(f"\n🎯 Rata-rata Akurasi (Mean CV Score) : {skor_cv.mean()*100:.2f}%")
print(f"📊 Standar Deviasi Skor              : ±{skor_cv.std()*100:.2f}% (Semakin kecil semakin stabil!)")


In [ ]:
# Visualisasi 1: Sebaran Skor Akurasi 5-Fold Cross-Validation
plt.figure(figsize=(9, 4.5))

putaran_labels = [f"Putaran {i}" for i in range(1, 6)]
bars = plt.bar(putaran_labels, skor_cv * 100, color='#3498db', edgecolor='black', alpha=0.85)

# Garis rata-rata acuan
mean_score = skor_cv.mean() * 100
plt.axhline(mean_score, color='crimson', linestyle='--', linewidth=2, label=f"Rata-rata: {mean_score:.2f}%")

# Arsiran rentang standar deviasi (+/- 1 std)
std_score = skor_cv.std() * 100
plt.axhspan(mean_score - std_score, mean_score + std_score, color='crimson', alpha=0.15, label=f"Rentang Stabilitas (±{std_score:.2f}%)")

# Tambahkan label angka di atas bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.3, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.title("Akurasi Tiap Putaran 5-Fold Cross Validation", fontsize=12, fontweight='bold')
plt.xlabel("Putaran Ujian (Fold)")
plt.ylabel("Akurasi Pengujian (%)")
plt.ylim(88, 100)
plt.legend(loc='lower right', frameon=True)
plt.grid(True, linestyle=':', alpha=0.5, axis='y')
plt.tight_layout()
plt.show()

print("💡 Cara Membaca Grafik:")
print("Model yang tangguh memiliki nilai bar yang merata dan berada dekat dengan garis putus-putus merah, membuktikan model tidak rapuh terhadap variasi data!")


## 🎛️ 3. Penyetelan Hyperparameter: GridSearchCV vs RandomizedSearchCV

### 🎚️ Apa Bedanya Parameter vs Hyperparameter?
* **Parameter**: Nilai internal yang dipelajari dan disesuaikan sendiri oleh model dari data (contoh: koefisien kemiringan garis regresi, bobot percabangan pohon).
* **Hyperparameter**: "Tombol setelan di luar kabin" yang harus kita atur sebelum model mulai belajar (contoh: `n_estimators`, `max_depth`, `min_samples_split`).

---

### 🔍 Dua Pendekatan Pencarian Parameter Optimal:
1. **GridSearchCV (Menyeluruh / Kisi Lengkap)**: Mencoba *seluruh kemungkinan kombinasi* parameter layaknya tabel perkalian. Dijamin menemukan kombinasi terbaik di dalam kisi, namun membutuhkan waktu komputasi yang lama jika kombinasinya ribuan.
2. **RandomizedSearchCV (Acak Cerdas)**: Mengambil sampel kombinasi secara acak dalam batas iterasi tertentu. Jauh lebih cepat (bisa 5x hingga 10x lebih hemat waktu) dengan hasil performa yang sering kali hampir identik!


In [ ]:
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Menentukan ruang hyperparameter yang ingin diuji
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 8],
    'min_samples_split': [2, 5]
}
# Total kombinasi kisi lengkap: 3 x 3 x 2 = 18 kombinasi (Dengan 3-fold CV = 54 kali pelatihan!)

# 1. Uji Coba GridSearchCV (Semua Kombinasi)
t0 = time.perf_counter()
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X, y)
t_grid = time.perf_counter() - t0

# 2. Uji Coba RandomizedSearchCV (Hanya Coba 6 Sampel Acak)
t0 = time.perf_counter()
rand_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=6,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
rand_search.fit(X, y)
t_rand = time.perf_counter() - t0

print("=== PERBANDINGAN EMPIRIS: GRID SEARCH VS RANDOM SEARCH ===")
print(f"1. GridSearchCV       : Akurasi = {grid_search.best_score_*100:.2f}% | Waktu = {t_grid:.2f} detik (18 kandidat)")
print(f"2. RandomizedSearchCV : Akurasi = {rand_search.best_score_*100:.2f}% | Waktu = {t_rand:.2f} detik (6 kandidat)")
print(f"Efisiensi Waktu       : RandomizedSearch {t_grid/t_rand:.1f}x lebih cepat!\n")

print("Kombinasi Parameter Terbaik (Grid Search):")
for k, v in grid_search.best_params_.items():
    print(f" - {k:<18}: {v}")

# Visualisasi 2: Peta Panas (Heatmap) Performa Hyperparameter
results_df = pd.DataFrame(grid_search.cv_results_)
pivot_table = results_df[results_df['param_min_samples_split'] == 2].pivot(
    index='param_max_depth',
    columns='param_n_estimators',
    values='mean_test_score'
)

plt.figure(figsize=(7, 4.5))
sns.heatmap(pivot_table * 100, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Akurasi CV (%)'})
plt.title("Peta Panas (Heatmap) Kombinasi Hyperparameter", fontsize=12, fontweight='bold')
plt.xlabel("n_estimators (Jumlah Pohon)", fontweight='bold')
plt.ylabel("max_depth (Kedalaman Pohon)", fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Kesimpulan Tuning:")
print("Warna biru tua pada heatmap menunjukkan 'zona manis' di mana akurasi model mencapai performa puncak!")


## 🔗 4. Membangun Pipeline Lengkap (*Scikit-Learn ColumnTransformer*)

Sekarang mari kita bangun **Pipeline Produksi Skala Penuh** menggunakan data Titanic (campuran angka, teks kategori, dan nilai kosong) sehingga model siap menerima data kotor apa pun secara aman!

---

### 🏗️ Arsitektur Lini Ban Berjalan Pipeline:
```
Data Mentah (Titanic)
     │
     ├── 🔢 Jalur Numerik   ──► SimpleImputer(median) ──► StandardScaler() ──┐
     │                                                                       ▼
     └── 🔤 Jalur Kategori  ──► SimpleImputer(modus)  ──► OneHotEncoder()    ──┼──► RandomForestClassifier
```
Dengan merangkai seluruh tahapan ke dalam satu objek `Pipeline`, kita **100% kebal dari Data Leakage** karena nilai median dan rumus scaling hanya dipelajari dari data latih!


In [ ]:
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Aktifkan tampilan diagram HTML interaktif Scikit-Learn
set_config(display="diagram")

# Data Simulasi Penumpang Kapal (Titanic-like Data)
df_titanic = pd.DataFrame({
    'Umur': [22, 38, 26, 35, np.nan, 54, 2, 27, np.nan, 40],
    'Tarif_Tiket': [7.25, 71.83, 7.92, 53.10, 8.05, 51.86, 21.07, 11.13, 30.07, 16.70],
    'Jenis_Kelamin': ['pria', 'wanita', 'wanita', 'wanita', 'pria', 'pria', 'pria', 'wanita', 'wanita', 'pria'],
    'Kelas_Kabin': ['Ekonomi', 'VIP', 'Ekonomi', 'VIP', 'Ekonomi', 'VIP', 'Bisnis', 'Bisnis', 'VIP', 'Bisnis'],
    'Selamat': [0, 1, 1, 1, 0, 0, 1, 1, 1, 0] # Target (0 = Tidak, 1 = Selamat)
})

X_raw = df_titanic[['Umur', 'Tarif_Tiket', 'Jenis_Kelamin', 'Kelas_Kabin']]
y_raw = df_titanic['Selamat']

# Tentukan kelompok kolom
fitur_numerik = ['Umur', 'Tarif_Tiket']
fitur_kategori = ['Jenis_Kelamin', 'Kelas_Kabin']

# 1. Pipa Jalur Numerik: Imputasi Median -> Standarisasi
pipa_numerik = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipa Jalur Kategori: Imputasi Modus -> One-Hot Encoding
pipa_kategori = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Penggabung Jalur: ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', pipa_numerik, fitur_numerik),
    ('cat', pipa_kategori, fitur_kategori)
])

# 4. Pipeline Final Terpadu: Preprocessing + Model Klasifikasi
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42))
])

# Latih seluruh rantai pipa hanya dengan SATU BARIS PERINTAH!
model_pipeline.fit(X_raw, y_raw)
print("✅ Seluruh rantai Pipeline berhasil dirakit dan dilatih dengan sempurna!")
model_pipeline


In [ ]:
# Tips Pro: Menyetel Hyperparameter LANGSUNG di Dalam Objek Pipeline
# Gunakan sintaks garis bawah ganda (__): 'nama_komponen__nama_parameter'
param_pipeline = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [2, 3]
}

grid_pipeline = GridSearchCV(model_pipeline, param_grid=param_pipeline, cv=2)
grid_pipeline.fit(X_raw, y_raw)

print("=== TUNING HYPERPARAMETER DI DALAM PIPELINE ===")
print(f"Skor Terbaik Pipeline : {grid_pipeline.best_score_*100:.2f}%")
print("Parameter Terbaik:")
for p, v in grid_pipeline.best_params_.items():
    print(f" - {p}: {v}")


## 💾 5. Serialisasi Model: Menyimpan & Memuat Kembali dengan `joblib`

Model yang sudah dilatih disimpan ke file biner (`.joblib`).  

### 🤔 Mengapa Memilih `joblib` Dibanding `pickle` Standar?
* Pustaka `pickle` adalah modul bawaan Python untuk menyimpan objek apa pun ke disk.
* Namun, model machine learning memuat **larik matriks NumPy berukuran raksasa** (seperti bobot ribuan pohon Random Forest).
* **`joblib`** dioptimalkan khusus oleh komunitas Scikit-Learn untuk mengompresi dan menyimpan larik NumPy hingga **10x lebih cepat** dengan ukuran file yang jauh lebih hemat!
* Siapa pun yang memiliki file `.joblib` ini dapat langsung memakainya di server web tanpa perlu melatih ulang!


In [ ]:
import joblib
import os

nama_file_model = "model_prediksi_titanic_pipeline.joblib"

# 1. Simpan Seluruh Pipeline ke Disk
joblib.dump(model_pipeline, nama_file_model)
ukuran_kb = os.path.getsize(nama_file_model) / 1024
print(f"💾 File berhasil disimpan ke disk : {nama_file_model} (Ukuran: {ukuran_kb:.2f} KB)")

# 2. Muat Ulang Pipeline dari Disk (Mensimulasikan lingkungan server aplikasi web baru)
pipeline_termuat = joblib.load(nama_file_model)
print("🔄 File berhasil dimuat kembali ke memori!")

# 3. Uji Prediksi Penumpang Baru (Data Mentah dengan Missing Values!)
penumpang_baru = pd.DataFrame([{
    'Umur': np.nan,            # Nilai hilang (akan otomatis diimputasi median oleh pipeline)
    'Tarif_Tiket': 65.50,
    'Jenis_Kelamin': 'wanita',
    'Kelas_Kabin': 'VIP'
}])

hasil_prediksi = pipeline_termuat.predict(penumpang_baru)[0]
peluang_selamat = pipeline_termuat.predict_proba(penumpang_baru)[0][1]

status_teks = "SELAMAT 🌟" if hasil_prediksi == 1 else "TIDAK SELAMAT ⚠️"
print("\n=== HASIL INFERENSI DATA PENUMPANG BARU ===")
print("Data Masukan    : Wanita, Tiket VIP ($65.50), Umur Tidak Diketahui")
print(f"Status Prediksi : {status_teks}")
print(f"Peluang Selamat : {peluang_selamat*100:.1f}%")


## 🧠 6. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep pipeline dan rekayasa model, lalu jalankan sel kode di bawah untuk memeriksa kunci jawabannya:

---
* **Soal 1**: Mengapa K-Fold Cross-Validation jauh lebih dianjurkan daripada pembagian data acak tunggal (*single train-test split*) saat membandingkan performa model?
* **Soal 2**: Apa perbedaan mendasar antara *Parameter* dan *Hyperparameter* pada model machine learning?
* **Soal 3**: Mengapa penggunaan `ColumnTransformer` dan `Pipeline` dari Scikit-Learn mutlak diperlukan untuk mencegah *Data Leakage*?
* **Soal 4**: Mengapa kita menyimpan seluruh objek `Pipeline` ke dalam `.joblib`, bukan hanya menyimpan objek model `RandomForestClassifier`-nya saja?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 9
kunci_jawaban_modul_9 = {
    "Soal 1": "Karena K-Fold menguji model di seluruh bagian data secara bergilir (5 atau 10 kali) dan menghitung skor rata-rata, sehingga performa yang dihasilkan tidak bergantung pada keberuntungan atau ketidakberuntungan pembagian acak tunggal.",
    "Soal 2": "Parameter adalah bobot internal yang dipelajari mesin sendiri dari data latih, sedangkan Hyperparameter adalah setelan konfigurasi eksternal yang ditentukan manusia sebelum pelatihan dimulai.",
    "Soal 3": "Karena Pipeline memastikan transformasi data (seperti mean imputer dan standard deviation scaler) hanya dihitung dari data latih pada setiap fold dan diterapkan secara terisolasi tanpa bocor ke data uji.",
    "Soal 4": "Jika hanya menyimpan modelnya saja, aplikasi web di masa depan akan gagal memproses data mentah pengguna yang belum di-imputasi, di-scale, dan di-one-hot encoding. Menyimpan seluruh Pipeline menjamin data mentah pengguna dapat langsung diproses otomatis tanpa kode transformasi tambahan."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 09 ===")
for soal, pembahasan in kunci_jawaban_modul_9.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 7. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Menguji Penumpang Laki-Laki Kelas Ekonomi
Ujilah model pipeline yang telah disimpan tadi untuk seorang penumpang baru:
* `Umur` = 25 tahun
* `Tarif_Tiket` = 7.50 USD
* `Jenis_Kelamin` = 'pria'
* `Kelas_Kabin` = 'Ekonomi'

Tugas Anda:
Jalankan inferensi dan cari tahu berapa persen peluang selamat penumpang tersebut menurut model AI!


In [ ]:
# Solusi Tantangan Mandiri Penumpang Ekonomi
penumpang_latihan = pd.DataFrame([{
    'Umur': 25.0,
    'Tarif_Tiket': 7.50,
    'Jenis_Kelamin': 'pria',
    'Kelas_Kabin': 'Ekonomi'
}])

pred_latihan = pipeline_termuat.predict(penumpang_latihan)[0]
prob_latihan = pipeline_termuat.predict_proba(penumpang_latihan)[0]

status_latihan = "SELAMAT 🌟" if pred_latihan == 1 else "TIDAK SELAMAT ⚠️"
print("Penumpang: Pria 25 tahun, Tiket Ekonomi ($7.50)")
print(f"Prediksi AI: {status_latihan} (Peluang Selamat: {prob_latihan[1]*100:.1f}%, Risiko Tidak Selamat: {prob_latihan[0]*100:.1f}%)")


## 🎯 8. Checklist Kelulusan Modul 09 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami mengapa notebook manual rentan *human error* dan bagaimana Pipeline menciptakan lini pabrik otomatis.
- [ ] Saya menguasai konsep dan cara kerja **K-Fold Cross-Validation** serta kapan harus memakai **StratifiedKFold**.
- [ ] Saya dapat membedakan konsep Parameter vs Hyperparameter, serta efisiensi kecepatan **RandomizedSearchCV** dibanding **GridSearchCV**.
- [ ] Saya memahami jebakan mematikan *Data Leakage* dan mengapa `Pipeline` adalah solusi mutlak pencegahannya.
- [ ] Saya mampu merakit **ColumnTransformer** dan **Pipeline** untuk memproses fitur numerik dan kategorikal sekaligus.
- [ ] Saya memahami sintaks penyetelan parameter langsung di dalam pipeline menggunakan garis bawah ganda (`estimator__parameter`).
- [ ] Saya memahami mengapa `joblib` lebih unggul daripada `pickle` untuk model data mining berbasis matriks NumPy.
- [ ] Saya berhasil mengekspor seluruh pipeline ke file biner `.joblib` dan memuatnya kembali untuk inferensi data baru.

---
### 🧭 Navigasi Modul Berikutnya:
Lanjutkan perjalanan Anda ke tonggak sejarah terpenting (*Fullstack Milestone*): Mengubah file artefak `.joblib` ini menjadi aplikasi web interaktif yang hidup, modern, dan fungsional menggunakan **Streamlit** langsung dari Google Colab hingga siap dibagikan ke seluruh dunia!

[![Open Next Module In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/10_Streamlit_Web_App_Colab.ipynb)  
👉 **[Menuju Modul 10: The Fullstack Milestone - Streamlit Web App](https://github.com/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/10_Streamlit_Web_App_Colab.ipynb)**
